In [0]:
#instalar dependencias
%pip install openpyxl


In [0]:
import hashlib
import pathlib
import shutil
import openpyxl

import pandas as pd
import requests

URL = ("https://www.ecuadorencifras.gob.ec/documentos/web-inec/Poblacion_y_Demografia/Proyecciones_Poblacionales/"
       "censo_2022/revision_2024_areas/Tabulados_Estimacion_Proyeccion_v2.xlsx")
# Huella del archivo revisado para este proyecto (Estimaciones y Proyecciones, revisión 2024 con áreas)
SHA256_REVISADO = "35dba440e116cad9e606d58960c1c27affc690392f6a29eacf651309bc5e01cd"
HOJA = "3.1"  # población cantonal por sexo y edad simple, 2010-2035
LANDING = pathlib.Path("/Volumes/vigia/bronze/landing/poblacion")
AGENTE = {"User-Agent": "vigia-dengue/0.1 (proyecto de portafolio sin fines comerciales)"}

In [0]:
original = LANDING / "original" / pathlib.PurePosixPath(URL).name
if not original.exists():
    respuesta = requests.get(URL, headers=AGENTE, timeout=300)
    respuesta.raise_for_status()
    original.parent.mkdir(parents=True, exist_ok=True)
    parcial = original.parent / f"{original.name}.parcial"
    parcial.write_bytes(respuesta.content)
    shutil.move(parcial, original)
contenido = original.read_bytes()
huella = hashlib.sha256(contenido).hexdigest()
print(f"{original.name}: {len(contenido) / 1024**2:.1f} MB · sha256 {huella}")
if huella != SHA256_REVISADO:
    print("Ojo: no es la versión revisada en este proyecto. Revisa los totales de la celda 4 antes de seguir")

In [0]:
def leer_hoja_cantonal(ruta, hoja=HOJA):
    """Población por cantón y año (hombres + mujeres, todas las edades) de la hoja cantonal del INEC

    El encabezado es la fila que dice Edad. Debajo van los años (una celda combinada por año: solo la
    primera tiene el número) y, en la fila siguiente, los sexos. Cada fila de datos es un cantón y una edad
    simple, de «0 a 100 y más"""
    d = pd.read_excel(ruta, sheet_name=hoja, header=None, dtype=object)
    texto = d.apply(lambda c: c.map(lambda v: "" if pd.isna(v) else str(v).strip()))
    fila = texto.index[(texto == "Edad").any(axis=1)][0]
    encabezado = texto.loc[fila]
    col_edad = encabezado[encabezado == "Edad"].index[0]
    col_canton = encabezado[encabezado.str.contains(r"C[oó]digo\s+cant[oó]n", case=False)].index[0]
    anios = d.loc[fila + 1].ffill()
    sexos = texto.loc[fila + 2]
    columnas = [c for c in d.columns if c > col_edad and sexos[c] in ("Hombre", "Mujer")]

    datos = d.loc[fila + 3:]
    es_edad = texto.loc[fila + 3:, col_edad].str.fullmatch(r"\d+(\.0)?( y más)?")
    codigo = pd.to_numeric(datos[col_canton], errors="coerce")
    filas = es_edad & codigo.notna()  # fuera quedan las notas y la fuente al pie
    datos = datos[filas].assign(dpa_canton=codigo[filas].astype(int).map("{:04d}".format))

    largo = datos.melt(id_vars="dpa_canton", value_vars=columnas, var_name="columna", value_name="poblacion")
    largo["anio"] = largo["columna"].map(anios).astype(float).astype(int)
    largo["poblacion"] = pd.to_numeric(largo["poblacion"])
    tabla = largo.groupby(["dpa_canton", "anio"], as_index=False)["poblacion"].sum()
    tabla["poblacion"] = tabla["poblacion"].round().astype("int64")
    tabla["dpa_provin"] = tabla["dpa_canton"].str[:2]
    return tabla[["dpa_provin", "dpa_canton", "anio", "poblacion"]]


cantonal = leer_hoja_cantonal(original)

In [0]:
por_anio = cantonal.groupby("anio").agg(cantones=("dpa_canton", "nunique"), poblacion=("poblacion", "sum"))
display(por_anio.loc[[2015, 2020, 2022, 2024, 2025, 2026]].reset_index())
assert cantonal["dpa_canton"].nunique() == 221, "el INEC trae 221 cantones (DPA del Censo 2022)"
assert por_anio.index.min() == 2010 and por_anio.index.max() == 2035
if huella == SHA256_REVISADO:
    # Totales del archivo revisado: los mismos que publica el INEC para el país
    assert por_anio.loc[2024, "poblacion"] == 17_966_573
    assert por_anio.loc[2025, "poblacion"] == 18_103_660
print("Controles en orden")

In [0]:
destino = LANDING / "revision_2024" / f"poblacion_cantonal_{huella[:8]}.csv"
destino.parent.mkdir(parents=True, exist_ok=True)
parcial = destino.parent / f"{destino.name}.parcial"
cantonal.assign(sha256=huella).to_csv(parcial, index=False)
shutil.move(parcial, destino)
print(f"Escrito {destino} ({len(cantonal)} filas)")